
# Scenario 31 — out-of-domain evaluation

Scores every available checkpoint on the **6,804** held-out scenario-31
samples, replacing the n=50 figures the generalisation finding currently rests
on.

Needs the preprocessed scenario-31 dataset produced by
`scenario31_preprocess.ipynb`, plus the checkpoint datasets. GPU, under an hour.


## 1. Setup

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd, torch

PROJECT = Path("/kaggle/working/project")
if not (PROJECT / "amber" / "model.py").exists():
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SathishAdithiyaaSV/MultiModal-Beam-Prediction.git", str(PROJECT)],
                       capture_output=True, text=True,
                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    assert r.returncode == 0, (r.stderr or "")[:400]
os.chdir(PROJECT); sys.path.insert(0, str(PROJECT))

from amber.ablation import config_label, parse_modalities
from amber.config import TrainConfig
from amber.dataset import AmberDataset
from amber.metrics import evaluate
from amber.predict import load_checkpoint
from amber.train import make_loader, pick_device, to_device
print("torch", torch.__version__, "cuda:", torch.cuda.is_available())

## 2. Locate the index and the checkpoints

In [ ]:
IN = Path("/kaggle/input")
cands = [d for d in (p.parent for p in IN.rglob("samples.csv"))
         if (d / "gps_norm.json").exists()]
INDEX_DIR = None
for d in cands:
    s = pd.read_csv(d / "samples.csv", usecols=["split"], nrows=5000)
    if (s["split"] == "scenario31_full").any():
        INDEX_DIR = d
        break
assert INDEX_DIR, f"no scenario-31 index among {cands}"
ROOT = INDEX_DIR.parent.parent
print("index:", INDEX_DIR)

SLUGS = ["gps", "gps_radar", "gps_lidar", "gps_radar_lidar", "gps_image",
         "gps_image_lidar", "gps_radar_image", "gps_radar_image_lidar"]
CKPTS = {}
for d, _dn, fn in os.walk(IN, followlinks=True):
    for f in fn:
        if f == "best.pt" and Path(d).name in SLUGS:
            CKPTS.setdefault(Path(d).name, Path(d) / f)
        elif f.endswith(".pt") and f[:-3] in SLUGS:
            CKPTS.setdefault(f[:-3], Path(d) / f)
print(f"{len(CKPTS)}/8 checkpoints:", ", ".join(sorted(CKPTS)))
assert CKPTS, "no checkpoints attached"

## 3. Evaluate

In [ ]:
device = pick_device()
train_cfg = TrainConfig()

@torch.no_grad()
def score(slug):
    model, cfg = load_checkpoint(CKPTS[slug], device)
    mods = parse_modalities(slug.split("_"))
    ds = AmberDataset(INDEX_DIR, "scenario31_full", cfg, root=ROOT, modalities=mods)
    loader = make_loader(ds, 32, shuffle=False, workers=2)
    lg, tg = [], []
    for b in loader:
        b = to_device(b, device)
        lg.append(model(b, use_cma=False).logits.float().cpu())
        tg.append(b["target"].cpu())
    lg, tg = torch.cat(lg), torch.cat(tg)
    lab = tg >= 0
    m = evaluate(lg[lab], tg[lab], ks=train_cfg.topk, delta=train_cfg.dba_delta)
    del model; torch.cuda.empty_cache()
    return m, int(lab.sum())

rows = []
for slug in sorted(CKPTS):
    t0 = time.time()
    m, n = score(slug)
    rows.append({"config": config_label(parse_modalities(slug.split("_"))),
                 "slug": slug, "Top-1": m["top1"], "Top-3": m["top3"],
                 "Top-5": m.get("top5", np.nan), "DBA": m["dba"], "n": n,
                 "has_camera": "image" in slug})
    print(f"{slug:<26} DBA {m['dba']:.4f}  Top-1 {m['top1']:.4f}  "
          f"n={n}  ({time.time()-t0:.0f}s)", flush=True)

res = pd.DataFrame(rows).sort_values("DBA", ascending=False)
OUT = Path("/kaggle/working/outputs/scenario31")
(OUT).mkdir(parents=True, exist_ok=True)
res.to_csv(OUT / "scenario31_results.csv", index=False)
print("\n" + res.round(4).to_string(index=False))


## 4. The generalisation finding, at 136× the evidence

The n=50 result was: camera configurations averaged 0.8764 in domain and
**0.0567** here, camera-free ones 0.7701 and 0.1940, with GPS alone the best of
all eight. If that ordering holds at n=6,804 it stops being an anecdote.


In [ ]:
cam = res[res.has_camera].DBA.mean()
nocam = res[~res.has_camera].DBA.mean()
print(f"camera configurations      mean DBA {cam:.4f}  (n={res.has_camera.sum()})")
print(f"camera-free configurations mean DBA {nocam:.4f}  (n={(~res.has_camera).sum()})")
print(f"\nbest overall: {res.iloc[0].config}  DBA {res.iloc[0].DBA:.4f}")
print(f"GPS alone rank: "
      f"{int((res.slug == 'gps').argmax()) + 1} of {len(res)}")

prior = {"camera_mean_n50": 0.0567, "camera_free_mean_n50": 0.1940}
print(f"\nversus the n=50 figures: camera {prior['camera_mean_n50']:.4f} -> {cam:.4f}, "
      f"camera-free {prior['camera_free_mean_n50']:.4f} -> {nocam:.4f}")
json.dump({"camera_mean": float(cam), "camera_free_mean": float(nocam),
           "n_samples": int(res.n.iloc[0]), "prior_n50": prior,
           "best_config": res.iloc[0].config},
          open(OUT / "generalisation_summary.json", "w"), indent=2)
for p in sorted(OUT.rglob("*")):
    print(" ", p.relative_to(OUT), f"({p.stat().st_size/1e3:.1f} KB)")